In [18]:
%load_ext autoreload
%autoreload 2

In [24]:
import torch as th
import os
import wandb

from fragnnet.runner import load_config, init_dataloader
from fragnnet.utils.script_utils import init_wandb_ckpt, run_inference
from fragnnet.graff.dataset import SpecMolAnnDataset
from fragnnet.graff.pl_model import GrAFFPL
from fragnnet.utils.spec_utils import calculate_sparse_cosine_similarity

In [2]:
th.cuda.is_available()

True

In [3]:
os.chdir("/fs01/home/adamo/frag-gnn")

In [4]:
config_d = load_config("config", "config/nist20v3-1_d4_wr00_scaffold/graff3_long/s0.yml")
config_d["compile"] = False
config_d["output_formula_str"] = False
for k in ["spec", "mol", "magma", "ann"]:
    key = f"{k}_params"
    config_d[key]["preprocess"] = False
    if k == "magma":
        config_d[key]["preload"] = False

In [5]:
print(config_d["auxiliary_scores"])

['cos_sim']


In [6]:
ds_cls = SpecMolAnnDataset
ds = ds_cls(split="test", **config_d)

<function fill_missing_nce at 0x7f415ee2ef80>: 100%|██████████| 262319/262319 [00:16<00:00, 15930.02it/s]


> warning: dropping spectra with huge mz diffs: 4707 annotations, 2591 spectra
> warning: dropping annotations with big mz diffs: 471919 annotations, 91523 spectra


/fs01/home/adamo/frag-gnn/src/frag_gnn/graff/dataset.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.isotope_df.loc[:,"has_isotopes"] = ann_df["ann_isotopes"].apply(any)


In [7]:
dl = init_dataloader(ds, config_d)

> init_dataloader for split test


In [8]:
model_cls = GrAFFPL
model = init_wandb_ckpt("w4mojs6u",last_ckpt=False,model_cls=model_cls,config_d=config_d)
model.to("cuda:0")
model.eval()

model.node_emb._orig_mod.0.weight  ==>  model.node_emb.0.weight
model.node_emb._orig_mod.0.bias  ==>  model.node_emb.0.bias
model.node_emb._orig_mod.3.weight  ==>  model.node_emb.3.weight
model.node_emb._orig_mod.3.bias  ==>  model.node_emb.3.bias
model.node_emb._orig_mod.4.weight  ==>  model.node_emb.4.weight
model.node_emb._orig_mod.4.bias  ==>  model.node_emb.4.bias
model.edge_emb._orig_mod.0.weight  ==>  model.edge_emb.0.weight
model.edge_emb._orig_mod.0.bias  ==>  model.edge_emb.0.bias
model.edge_emb._orig_mod.3.weight  ==>  model.edge_emb.3.weight
model.edge_emb._orig_mod.3.bias  ==>  model.edge_emb.3.bias
model.edge_emb._orig_mod.4.weight  ==>  model.edge_emb.4.weight
model.edge_emb._orig_mod.4.bias  ==>  model.edge_emb.4.bias
model.signnet._orig_mod.phi.0.weight  ==>  model.signnet.phi.0.weight
model.signnet._orig_mod.phi.0.bias  ==>  model.signnet.phi.0.bias
model.signnet._orig_mod.phi.1.weight  ==>  model.signnet.phi.1.weight
model.signnet._orig_mod.phi.1.bias  ==>  model.sig

GrAFFPL(
  (model): GrAFFModel(
    (onehot): CanonicalOneHot()
    (node_emb): Sequential(
      (0): Linear(in_features=174, out_features=1024, bias=True)
      (1): SiLU(inplace=True)
      (2): Dropout(p=0.0, inplace=False)
      (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (4): Linear(in_features=1024, out_features=1024, bias=True)
    )
    (edge_emb): Sequential(
      (0): Linear(in_features=17, out_features=1024, bias=True)
      (1): SiLU(inplace=True)
      (2): Dropout(p=0.0, inplace=False)
      (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (4): Linear(in_features=1024, out_features=1024, bias=True)
    )
    (signnet): SignNet(
      (phi): Sequential(
        (0): Linear(in_features=2, out_features=32, bias=True)
        (1): Linear(in_features=32, out_features=32, bias=True)
        (2): SiLU(inplace=True)
        (3): Dropout(p=0.0, inplace=False)
        (4): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (5): Lin

In [9]:
output_subset = [
    "true_mzs",
    "true_logprobs",
    "true_batch_idxs",
    "pred_mzs",
    "pred_logprobs",
    "pred_batch_idxs",
    "true_oos_prob",
    "oos_prob",
]
output_subset = set(output_subset)
output_subset.union(model.metric_names)

{'cos_sim_0.01',
 'loss',
 'oos_prob',
 'pred_batch_idxs',
 'pred_logprobs',
 'pred_mzs',
 'primary_loss',
 'true_batch_idxs',
 'true_logprobs',
 'true_mzs',
 'true_oos_prob'}

In [10]:
vals = run_inference(
    dl, 
    model, 
    "cuda:0", 
    "val", 
    int(1e6),
    False, 
    output_subset=None, 
    untransform_spec=False
)

>>> Iterating over data


100%|██████████| 27/27 [03:47<00:00,  8.43s/it]


> num_datapoints = 3341
> num_nodes = 0
> num_formulas = 0
> num_nb_nodes = 0


In [11]:
vals.keys()

dict_keys(['unique_id', 'smiles', 'mean_loss', 'total_loss', 'total_weight', 'spec_per_group', 'spec_per_mol', 'group_per_mol', 'true_mzs', 'true_logprobs', 'true_batch_idxs', 'true_prec_mzs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'loss', 'primary_loss', 'cos_sim_0.01'])

In [14]:
print(vals["cos_sim_0.01"].mean())

tensor(0.5228)


In [22]:
cos_sim = calculate_sparse_cosine_similarity(
    vals["true_mzs"], 
    vals["true_logprobs"].exp(), 
    vals["true_batch_idxs"],
    vals["pred_mzs"], 
    vals["pred_logprobs"].exp(),
    vals["pred_batch_idxs"],
    mz_max=1500.,
    mz_bin_res=0.01,
)
print(cos_sim.mean())

tensor(0.5224)


In [27]:
cos_sim_log = calculate_sparse_cosine_similarity(
    vals["true_mzs"], 
    vals["true_logprobs"], 
    vals["true_batch_idxs"],
    vals["pred_mzs"], 
    vals["pred_logprobs"],
    vals["pred_batch_idxs"],
    mz_max=1500.,
    mz_bin_res=0.01,
    log_ints=True
)
print(cos_sim_log.mean())

tensor(0.5224)
